# 04 — Build the Al–Cu Wobbling Welding Dataset

This notebook standardizes the supplied wobbling-welding workbook while preserving all 36 experimental records, missing measurements, and source notes. It does not impute values, remove records, or average replicates.

**Input:** `data/raw/Al-Cu_Wobbling_Welding.xlsx`  
**Output:** `data/processed/alcu_wobbling_welding_standardized.csv`


In [ ]:
from pathlib import Path
import pandas as pd

RAW_FILE = Path('data/raw/Al-Cu_Wobbling_Welding.xlsx')
OUT_DIR = Path('data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_FILE.exists():
    raise FileNotFoundError(f'Cannot find {RAW_FILE}. Upload the raw workbook into data/raw first.')

raw = pd.read_excel(RAW_FILE, sheet_name='Data')
print('Rows:', len(raw))
print('Columns:', raw.columns.tolist())
display(raw.head())


## 1. Standardize column names and preserve source notes

The source workbook includes an unnamed column containing the note `Specimen cut`. We retain it as `source_note` rather than discarding it.

In [ ]:
required = {
    'Laser Power [W]': 'laser_power_W',
    'Tangential wobbling speed [mm/s]': 'tangential_wobbling_speed_mm_s',
    'Wobbling diameter [mm]': 'wobbling_diameter_mm',
    'Bead with (face) [mm]': 'bead_width_face_mm',
    'Bead width (interface) [mm]': 'bead_width_interface_mm',
    'Penetration depth [mm]': 'penetration_depth_mm',
}
missing = [c for c in required if c not in raw.columns]
if missing:
    raise ValueError(f'Missing expected source columns: {missing}')

df = raw[list(required)].rename(columns=required).copy()

# Preserve the original explanatory note, including blank notes.
note_col = 'Unnamed: 6'
if note_col in raw.columns:
    df['source_note'] = raw[note_col].fillna('').astype(str).str.strip()
else:
    df['source_note'] = ''

df.insert(0, 'source_id', 'alcu_wobbling_welding_dataset')
df.insert(1, 'study_id', 'source_dataset_02')
df.insert(2, 'record_id', [f'WOB_{i+1:03d}' for i in range(len(df))])
condition_cols = ['laser_power_W', 'tangential_wobbling_speed_mm_s', 'wobbling_diameter_mm']
df['replicate_id'] = df.groupby(condition_cols).cumcount() + 1

width_cols = ['bead_width_face_mm', 'bead_width_interface_mm']
df['width_measurements_missing'] = df[width_cols].isna().any(axis=1)
df['measurement_status'] = 'width measurements available'
df.loc[df['source_note'].eq('Specimen cut'), 'measurement_status'] = 'specimen cut; width measurements unavailable'
df.loc[df['width_measurements_missing'] & df['source_note'].eq(''), 'measurement_status'] = 'width measurements missing; reason not stated in workbook'

display(df.head(12))


## 2. Audit missing measurements and experimental repeats

Check the workbook's missing-value pattern and condition counts. Missing widths are not treated as zero.

In [ ]:
print('Dataset shape:', df.shape)
print('\nMissing values by column:')
display(df.isna().sum().rename('missing_count').to_frame())

print('\nMeasurement status counts:')
display(df['measurement_status'].value_counts(dropna=False).rename_axis('status').to_frame('records'))

print('\nOriginal source notes:')
display(df['source_note'].value_counts(dropna=False).rename_axis('source_note').to_frame('records'))

print('\nReplicate count by process condition:')
display(df.groupby(condition_cols).size().rename('record_count').reset_index())

print('\nDuplicate complete standardized rows:', int(df.duplicated().sum()))
print('No records have been deleted or averaged.')


## 3. Save the standardized dataset

The CSV is a derived working file. The raw workbook remains unchanged.

In [ ]:
output_file = OUT_DIR / 'alcu_wobbling_welding_standardized.csv'
df.to_csv(output_file, index=False)
check = pd.read_csv(output_file)
print('Saved:', output_file)
print('Rows saved:', len(check))
print('Missing values in saved file:', int(check.isna().sum().sum()))


## 4. Download the standardized CSV from Colab

This downloads the output from the current Colab runtime. It does not automatically commit it to GitHub.

In [ ]:
from google.colab import files
files.download(str(output_file))
